In [5]:
import pandas as pd

df = pd.read_csv('datasets/csic_database.csv')

print("Shape:", df.shape)
print("\nEXACT column names in this file:")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}. [{col}]")

print("\nFirst 2 rows:")
df.head(2)

Shape: (61065, 17)

EXACT column names in this file:
   0. [Unnamed: 0]
   1. [Method]
   2. [User-Agent]
   3. [Pragma]
   4. [Cache-Control]
   5. [Accept]
   6. [Accept-encoding]
   7. [Accept-charset]
   8. [language]
   9. [host]
  10. [cookie]
  11. [content-type]
  12. [connection]
  13. [lenght]
  14. [content]
  15. [classification]
  16. [URL]

First 2 rows:


,Unnamed: 0,Method,User-Agent,Pragma,Cache-Control,Accept,Accept-encoding,Accept-charset,language,host,cookie,content-type,connection,lenght,content,classification,URL
0,Normal,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=1F767F17239C9B670A39E9B10C3825F4,NaN,close,NaN,NaN,0,http://localhost:8080/tienda1/index.jsp HTTP/1.1
1,Normal,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,NaN,close,NaN,NaN,0,http://localhost:8080/tienda1/publico/anadir.j...


In [6]:
print("=" * 50)
print(f"ROWS: {df.shape[0]:,}   COLUMNS: {df.shape[1]}")
print("=" * 50)

print("\nHow full is each column:")
for col in df.columns:
    filled = df[col].notna().sum()
    pct    = filled / len(df) * 100
    bar    = "█" * int(pct / 10)
    print(f"  {col:20s}  {pct:5.1f}%  {bar}")

ROWS: 61,065   COLUMNS: 17

How full is each column:
  Unnamed: 0            100.0%  ██████████
  Method                100.0%  ██████████
  User-Agent            100.0%  ██████████
  Pragma                100.0%  ██████████
  Cache-Control         100.0%  ██████████
  Accept                 99.3%  █████████
  Accept-encoding       100.0%  ██████████
  Accept-charset        100.0%  ██████████
  language              100.0%  ██████████
  host                  100.0%  ██████████
  cookie                100.0%  ██████████
  content-type           29.4%  ██
  connection            100.0%  ██████████
  lenght                 29.4%  ██
  content                29.4%  ██
  classification        100.0%  ██████████
  URL                   100.0%  ██████████


In [7]:
print("=" * 50)
print("LABEL DISTRIBUTION")
print("=" * 50)

# In this file: 0 = Normal, 1 = Anomalous
counts = df['classification'].value_counts()
print("Raw values:", counts.to_dict())
print()

labels = {0: 'Normal', 1: 'Anomalous'}
for val, count in counts.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    name = labels.get(val, str(val))
    print(f"  {name:12s} (={val})  {count:6,}  ({pct:.1f}%)  {bar}")

print("\n" + "=" * 50)
print("HTTP METHOD DISTRIBUTION")
print("=" * 50)
print(df['Method'].value_counts())

print("\n" + "=" * 50)
print("METHOD SPLIT BY LABEL")
print("=" * 50)
print(df.groupby(['classification', 'Method']).size().unstack(fill_value=0))

LABEL DISTRIBUTION
Raw values: {0: 36000, 1: 25065}

  Normal       (=0)  36,000  (59.0%)  █████████████████████████████
  Anomalous    (=1)  25,065  (41.0%)  ████████████████████

HTTP METHOD DISTRIBUTION
Method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64

METHOD SPLIT BY LABEL
Method            GET  POST  PUT
classification                  
0               28000  8000    0
1               15088  9580  397


In [8]:
print("=" * 50)
print("NULL COUNT PER COLUMN")
print("=" * 50)

null_report = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().sum() / len(df) * 100).round(1)
}).sort_values('null_count', ascending=False)

print(null_report.to_string())
print(f"\nTotal null cells: {df.isnull().sum().sum():,}")

print("\n--- NOTE on high nulls ---")
print("'content' and 'lenght' have high nulls because")
print("GET requests have no body — this is CORRECT HTTP behaviour.")

NULL COUNT PER COLUMN
                 null_count  null_pct
content               43088      70.6
lenght                43088      70.6
content-type          43088      70.6
Accept                  397       0.7
Unnamed: 0                0       0.0
host                      0       0.0
classification            0       0.0
connection                0       0.0
cookie                    0       0.0
language                  0       0.0
Method                    0       0.0
Accept-charset            0       0.0
Accept-encoding           0       0.0
Cache-Control             0       0.0
Pragma                    0       0.0
User-Agent                0       0.0
URL                       0       0.0

Total null cells: 129,661

--- NOTE on high nulls ---
'content' and 'lenght' have high nulls because
GET requests have no body — this is CORRECT HTTP behaviour.


In [9]:
print("=" * 60)
print("5 NORMAL REQUESTS (classification = 0)")
print("=" * 60)
normal_sample = df[df['classification'] == 0].head(5)
for i, row in normal_sample.iterrows():
    print(f"\n  Method : {row['Method']}")
    print(f"  URL    : {str(row['URL'])[:80]}")
    body = str(row.get('content', ''))
    if body and body != 'nan':
        print(f"  Body   : {body[:80]}")

print("\n" + "=" * 60)
print("5 ATTACK REQUESTS (classification = 1)")
print("=" * 60)
attack_sample = df[df['classification'] == 1].head(5)
for i, row in attack_sample.iterrows():
    print(f"\n  Method : {row['Method']}")
    print(f"  URL    : {str(row['URL'])[:80]}")
    body = str(row.get('content', ''))
    if body and body != 'nan':
        print(f"  Body   : {body[:80]}")

5 NORMAL REQUESTS (classification = 0)

  Method : GET
  URL    : http://localhost:8080/tienda1/index.jsp HTTP/1.1

  Method : GET
  URL    : http://localhost:8080/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=1

  Method : POST
  URL    : http://localhost:8080/tienda1/publico/anadir.jsp HTTP/1.1
  Body   : id=3&nombre=Vino+Rioja&precio=100&cantidad=55&B1=A%F1adir+al+carrito

  Method : GET
  URL    : http://localhost:8080/tienda1/publico/autenticar.jsp?modo=entrar&login=choong&pw

  Method : POST
  URL    : http://localhost:8080/tienda1/publico/autenticar.jsp HTTP/1.1
  Body   : modo=entrar&login=choong&pwd=d1se3ci%F3n&remember=off&B1=Entrar

5 ATTACK REQUESTS (classification = 1)

  Method : GET
  URL    : http://localhost:8080/tienda1/publico/anadir.jsp?id=2&nombre=Jam%F3n+Ib%E9rico&p

  Method : POST
  URL    : http://localhost:8080/tienda1/publico/anadir.jsp HTTP/1.1
  Body   : id=2&nombre=Jam%F3n+Ib%E9rico&precio=85&cantidad=%27%3B+DROP+TABLE+usuarios%3B+S

  Method : 

In [10]:
df['url_length']     = df['URL'].astype(str).str.len()
df['content_length'] = df['content'].fillna('').astype(str).str.len()

print("URL LENGTH: Normal(0) vs Attack(1)")
print(df.groupby('classification')['url_length'].describe().round(1))

print("\nCONTENT LENGTH: Normal(0) vs Attack(1)")
print(df.groupby('classification')['content_length'].describe().round(1))

print("\nTOP 3 LONGEST ATTACK URLs:")
top = df[df['classification'] == 1].nlargest(3, 'url_length')
for _, row in top.iterrows():
    print(f"\n  Length {row['url_length']}: {str(row['URL'])[:100]}")

print("\nCOLUMNS WITH ONLY 1 UNIQUE VALUE (useless for ML):")
for col in df.columns:
    n = df[col].nunique()
    if n <= 2 and col not in ['classification', 'Method']:
        print(f"  {col:25s} → {n} unique values  DROP")

URL LENGTH: Normal(0) vs Attack(1)
                  count   mean   std   min   25%   50%    75%    max
classification                                                      
0               36000.0   79.0  59.4  48.0  57.0  58.5   67.0  367.0
1               25065.0  106.6  91.6  31.0  57.0  64.0  122.0  895.0

CONTENT LENGTH: Normal(0) vs Attack(1)
                  count  mean   std  min  25%  50%   75%    max
classification                                                 
0               36000.0  20.3  59.0  0.0  0.0  0.0   0.0  307.0
1               25065.0  48.6  92.1  0.0  0.0  0.0  62.0  836.0

TOP 3 LONGEST ATTACK URLs:

  Length 895: http://localhost:8080/tienda1/publico/registro.jsp?modo=registro&login=erdinc&password=3e39a62r&nomb

  Length 895: http://localhost:8080/tienda1/miembros/editar.jsp?modo=registro&login=sessionid%3D12312312%26+userna

  Length 886: http://localhost:8080/tienda1/publico/registro.jsp?modo=registro&login=zalman&password=*diet%E9tico&

COLUMNS WITH ONL

## Day 11 Summary — csic_database.csv

### Real column names (important for Day 12)
- Target column : `classification`  (0=Normal, 1=Anomalous)
- Body column   : `content`         (NOT payload)
- Length column : `lenght`          (typo in dataset — no 'n')
- Request path  : `URL`
- Verb          : `Method`

### Label distribution
- Normal    (0): 36,000 rows (58.9%)
- Anomalous (1): 25,065 rows (41.1%)

### Method distribution
- GET: 43,088  POST: 17,580  PUT: 397

### Columns to DROP on Day 12 (same value in every row)
User-Agent, Pragma, Cache-Control, Accept, Accept-encoding,
Accept-charset, language, host, connection, Unnamed: 0

### Features to KEEP and engineer on Day 12
- url_length      (from URL column)
- content_length  (from content column)
- method_encoded  (GET=0, POST=1, PUT=2)
- has_content     (1 if content exists, 0 if not)
- lenght          (the numeric column — fill nulls with 0)